## DATABRICKS TRAINING SESSIONS
<br>**DATE: 2026-08-31**
<br>**TOPIC: Medallion Architecture**

#Staging Directory

In [0]:
catalog_name = dbutils.widgets.get("catalog_name")
spark.sql(f" USE CATALOG {catalog_name}")

In [0]:
%sql

-- USE CATALOG wns24082026;
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;

In [0]:
%skip
from shutil import move

files = ['orders_001.json','orders_002.json','orders_003.json','orders_004.json']
source_folder = "/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/orders/"
target_folder = "/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging"

for file_name in files:
    file_path = f"{source_folder}{file_name}"
    df = spark.read.json(file_path)
    move(file_path, f"{target_folder}/{file_name}")


In [0]:
cust_path = "/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/customers/customers_001.csv"
customer_df = spark.read.csv(cust_path, header=True, inferSchema=True)

products_path = "/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/products"
products_df = spark.read.json(products_path)

orders_path = (
    "/Volumes/wns24082026/quickstart_schema/sandbox/datasets/e-commerce/staging/orders"
)
orders_df = spark.read.json(orders_path)

#Bronze Layer

In [0]:
%skip
dict_df_table = {"customer_df":"bronze.customers",
                 "products_df":"bronze.products",
                 "customer_df":"bronze.orders"}

def savetable(dict_df_table):
    for df_name, table_name in dict_df_table.items():
        eval(df_name).write.saveAsTable(table_name, mode="OVERWRITE")

In [0]:
customer_df.write.saveAsTable('bronze.customers', mode = "OVERWRITE")
products_df.write.saveAsTable('bronze.products', mode = "OVERWRITE")
orders_df.write.saveAsTable('bronze.orders',  mode = "OVERWRITE")

#Silver Layer

In [0]:
from pyspark.sql.functions import col

spark.read.table("wns24082026.bronze.orders").filter(
    col("order_id").isNotNull()
).withColumn("total_price", col("qty") * col("price")).write.saveAsTable(
    "silver.orders", mode = "OVERWRITE"
)

#Gold Layer

In [0]:
from pyspark.sql.functions import col, sum

spark.read.table("wns24082026.silver.orders").groupBy("item_id").agg(
    sum("total_price").alias("total_price")
).write.saveAsTable("gold.revenue_by_product", mode = "APPEND")